# MOD-07 · NB-04 — Disease Mapping & Bayesian Spatial Smoothing
### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
---
**Learning objectives**
- Understand the small-area estimation problem: unstable rates in sparse populations
- Implement empirical Bayes smoothing (James-Stein, gamma-Poisson)
- Conceptualise the BYM (Besag-York-Mollié) CAR model structure
- Compare raw SMR vs smoothed estimates visually and statistically
- Apply MCMC-based spatial smoothing using PyMC (when available)
- Assess shrinkage patterns in Bayesian disease mapping

**Estimated time:** 3 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `matplotlib`, `pymc` (optional)


## 1. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.ndimage import gaussian_filter; from scipy.spatial import cKDTree
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty=10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
pm25=8+4*np.random.gamma(2,1,N); population=np.random.lognormal(10.5,1.0,N).astype(int)
def sp(vals,sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk=sp(np.random.normal(0,1,N))
cvd_rate=(180+1.2*pct_poverty+0.5*pm25+15*spatial_risk+np.random.normal(0,12,N))
expected=(cvd_rate/100000)*population*5
observed=np.random.poisson(expected).astype(int)
smr=observed/expected.clip(1)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pm25":pm25,"population":population,
    "cvd_rate":cvd_rate,"observed":observed,"expected":expected,"smr":smr})
print(f"N={N} | CVD={cvd_rate.mean():.1f} | SMR range=[{smr.min():.2f},{smr.max():.2f}]")

## 2. The Small-Area Estimation Problem

**Problem:** Observed SMR = O_i / E_i is unstable for areas with small populations.

A county with expected deaths E=2 and observed O=4 has SMR=2.0, suggesting 100% excess mortality — but this could easily be random variation.

**Key insight:** Raw rates have high variance in low-population areas. Bayesian methods "borrow strength" from neighbouring areas and the overall mean.

| Method | Shrinkage target | Spatially informed? |
|--------|-----------------|---------------------|
| Raw SMR | None | No |
| Empirical Bayes (EB) | Overall mean | No |
| BYM/CAR | Spatial neighbours | Yes |
| Full Bayesian MCMC | Prior + data | Yes (BYM) |


In [ ]:
# Visualise the small-area problem
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw SMR
sc = axes[0].scatter(df.lon, df.lat, c=df.smr, cmap="RdBu_r",
                      vmin=0.5, vmax=1.8, s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc, ax=axes[0], label="Raw SMR")
axes[0].set_title("Raw SMR — noisy in small areas")

# Population (denominator)
sc2 = axes[1].scatter(df.lon, df.lat, c=np.log(df.population), cmap="Blues",
                       s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc2, ax=axes[1], label="log(Population)")
axes[1].set_title("Log population (denominator size)")

# Expected deaths
sc3 = axes[2].scatter(df.lon, df.lat, c=df.expected, cmap="YlOrRd",
                       s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc3, ax=axes[2], label="Expected deaths (5-yr)")
axes[2].set_title("Expected deaths — reliability indicator")

for ax in axes: ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
plt.suptitle("Small-Area Problem: Raw SMR Most Unreliable Where Expected Counts Are Small", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/small_area_problem.png", bbox_inches="tight"); plt.show()

# Correlation: instability (high CV) vs small expected
cv_smr = np.where(df.expected < 5, "< 5 expected", ">= 5 expected")
print("SMR variance by expected count group:")
for grp in ["< 5 expected", ">= 5 expected"]:
    mask = cv_smr == grp
    print(f"  {grp:20s}: n={mask.sum():3d} | SMR mean={df.smr[mask].mean():.3f} | SD={df.smr[mask].std():.3f}")


## 3. Empirical Bayes Smoothing

In [ ]:
# Gamma-Poisson (conjugate) Empirical Bayes
# Model: O_i | theta_i ~ Poisson(E_i * theta_i)
#        theta_i ~ Gamma(alpha, beta)
# Posterior mean: E[theta_i|O_i] = (O_i + alpha) / (E_i + beta)
# Shrinkage toward overall rate: alpha/beta

# Estimate hyperparameters from marginal distribution
# Using method of moments: E[SMR] = alpha/beta, Var[SMR] = alpha/beta^2 (overdispersion)
smr_mean = df.smr.mean()
smr_var  = df.smr.var()
# Extra-Poisson variance adjustment
extra_var = max(smr_var - smr_mean/df.expected.mean(), 0.01)
alpha_hat = smr_mean**2 / extra_var
beta_hat  = smr_mean / extra_var

print(f"EB Hyperparameters: alpha={alpha_hat:.3f}, beta={beta_hat:.3f}")
print(f"Prior mean (overall rate): {alpha_hat/beta_hat:.3f}")

# Smoothed estimates
df["smr_eb"] = (df.observed + alpha_hat) / (df.expected + beta_hat)
df["smr_shrinkage"] = 1 - (df.expected / (df.expected + beta_hat))  # how much shrinkage

print(f"\nSmoothing results:")
print(f"  Raw SMR   : mean={df.smr.mean():.3f}  SD={df.smr.std():.3f}  range=[{df.smr.min():.2f},{df.smr.max():.2f}]")
print(f"  EB Smooth : mean={df.smr_eb.mean():.3f}  SD={df.smr_eb.std():.3f}  range=[{df.smr_eb.min():.2f},{df.smr_eb.max():.2f}]")
print(f"  Mean shrinkage: {df.smr_shrinkage.mean()*100:.1f}%")
print(f"  High shrinkage (>50%): {(df.smr_shrinkage>0.5).sum()} counties (low population)")


In [ ]:
# Compare raw SMR vs EB smoothed
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vmin, vmax = 0.5, 1.8

sc1 = axes[0].scatter(df.lon, df.lat, c=df.smr, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc1, ax=axes[0], label="Raw SMR"); axes[0].set_title("Raw SMR")

sc2 = axes[1].scatter(df.lon, df.lat, c=df.smr_eb, cmap="RdBu_r",
                       vmin=vmin, vmax=vmax, s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc2, ax=axes[1], label="EB-Smoothed SMR"); axes[1].set_title("Empirical Bayes Smoothed SMR")

sc3 = axes[2].scatter(df.lon, df.lat, c=df.smr_shrinkage, cmap="Blues",
                       s=150, alpha=0.85, edgecolors="white")
plt.colorbar(sc3, ax=axes[2], label="Shrinkage fraction")
axes[2].set_title("Shrinkage: 1 = fully shrunk to mean\n(small areas shrunk more)")

for ax in axes: ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
plt.suptitle("Empirical Bayes Smoothing: Stabilising Small-Area Rates", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/eb_smoothing.png", bbox_inches="tight"); plt.show()


## 4. CAR/BYM Model — Conceptual Framework

The **Besag-York-Mollié (BYM)** model decomposes the random effect into:

```
Y_i | theta_i ~ Poisson(E_i * theta_i)
log(theta_i) = mu + u_i + v_i
```

Where:
- `u_i` = **spatially structured** random effect (ICAR prior — neighbours pull toward each other)
- `v_i` = **unstructured** random effect (i.i.d. normal — pure noise)
- `mu`  = overall log-relative risk

**ICAR prior for u_i:**
```
u_i | u_{-i} ~ Normal(mean(u_j for j in neighbours_i), sigma²_u / n_i)
```

This is equivalent to a joint Gaussian distribution with precision matrix Q = D - W  
where D is the diagonal matrix of neighbour counts and W is the adjacency matrix.


In [ ]:
# Simulate BYM-like spatial smoothing without MCMC
# (Approximate BYM via spatial filter + Poisson regression)

def build_queen_binary(row_idx, col_idx, nrow, ncol):
    N=len(row_idx); W=np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i!=j and max(abs(row_idx[i]-row_idx[j]),abs(col_idx[i]-col_idx[j]))<=1:
                W[i,j]=1
    return W

W_bin = build_queen_binary(df.row.values, df.col.values, NROW, NCOL)
D_diag = W_bin.sum(axis=1)  # degree matrix diagonal

# Intrinsic CAR (ICAR) approximate random effects
# via eigendecomposition of graph Laplacian Q = D - W
Q = np.diag(D_diag) - W_bin
# Spatial basis functions from eigenvectors of Q (Moran eigenvectors)
eigenvalues, eigenvectors = np.linalg.eigh(Q)
# Small eigenvalues correspond to smooth spatial patterns
spatial_basis = eigenvectors[:, :10]  # first 10 eigenvectors (smoothest patterns)

from sklearn.linear_model import Ridge
# Log-offset Poisson: log(O_i/E_i) ~ X + spatial_basis
log_smr = np.log(df.smr.clip(0.01, 5))
X_bym = np.column_stack([spatial_basis])
# Ridge regression on log(SMR) as approximation
ridge = Ridge(alpha=1.0)
ridge.fit(X_bym, log_smr)
log_smr_hat = ridge.predict(X_bym)
df["smr_bym_approx"] = np.exp(log_smr_hat)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vmin, vmax = 0.6, 1.7
for ax, col, title in [
    (axes[0], "smr",            "Raw SMR"),
    (axes[1], "smr_eb",         "EB Smoothed SMR"),
    (axes[2], "smr_bym_approx", "Approx BYM Smoothed (ICAR basis)"),
]:
    sc = ax.scatter(df.lon, df.lat, c=df[col], cmap="RdBu_r",
                    vmin=vmin, vmax=vmax, s=150, alpha=0.85, edgecolors="white")
    plt.colorbar(sc, ax=ax, label="SMR"); ax.set_title(title)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
plt.suptitle("Disease Mapping: Raw SMR → EB → BYM-Approximate Smoothing", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/bym_smoothing.png", bbox_inches="tight"); plt.show()

for name, col in [("Raw",df.smr),("EB",df.smr_eb),("BYM-approx",df.smr_bym_approx)]:
    print(f"  {name:12s}: SD={col.std():.4f} | Range=[{col.min():.2f},{col.max():.2f}]")


## 5. PyMC BYM Starter Code

In [ ]:
# Full Bayesian BYM using PyMC (run if pymc available)
PYMC_TEMPLATE = ''' import pymc as pm, numpy as np, pytensor.tensor as pt  # Adjacency matrix as sparse structure # W_bin: binary adjacency (queen contiguity) N = len(df) node1, node2 = np.where(np.triu(W_bin, 1) > 0)  # upper triangle edges  with pm.Model() as bym_model: # Hyperpriors sigma_u = pm.HalfNormal("sigma_u", sigma=1)   # spatial SD sigma_v = pm.HalfNormal("sigma_v", sigma=1)   # unstructured SD mu      = pm.Normal("mu", mu=0, sigma=1)       # overall log-RR  # Spatial random effect (ICAR) — sum-to-zero constraint u_raw = pm.Flat("u_raw", shape=N) u = pm.Deterministic("u", u_raw - pt.mean(u_raw)) # ICAR prior: penalises differences between neighbours icar_logp = -0.5 * sigma_u**(-2) * pt.sum((u[node1] - u[node2])**2) pm.Potential("icar", icar_logp)  # Unstructured random effect v = pm.Normal("v", mu=0, sigma=sigma_v, shape=N)  # Linear predictor log_theta = mu + sigma_u * u + v theta     = pm.math.exp(log_theta)  # Likelihood obs = pm.Poisson("obs", mu=df["expected"].values * theta, observed=df["observed"].values)  # Inference trace = pm.sample(2000, tune=1000, target_accept=0.9, cores=2, random_seed=42)  # Extract smoothed SMR smr_bym_full = np.exp(trace.posterior["log_theta"].mean(dim=["chain","draw"]).values) '''
print("PyMC BYM template saved.")
print("To run: pip install pymc pytensor")
print()
print("Key BYM parameters to monitor:")
print("  sigma_u: spatial random effect SD (larger = more spatial clustering)")
print("  sigma_v: unstructured random effect SD (larger = more pure noise)")
print("  u_i: spatially structured relative risk (map this for disease clusters)")
print("  theta_i: total relative risk = exp(mu + sigma_u*u_i + v_i)")
print()
print("Model selection: DIC or WAIC comparing:")
print("  Null model (mu only) vs EB (v only) vs BYM (u + v)")


## 6. Knowledge Check
1. A county has O=1 death, E=0.8, so SMR=1.25. The EB-smoothed estimate is 1.02. Why does EB shrink this so aggressively?
2. BYM model sigma_u >> sigma_v. What does this imply about the nature of spatial clustering?
3. A researcher uses raw SMR to rank counties for intervention. What are the consequences of not smoothing?
4. Moran's I for raw SMR = 0.41 but for EB-smoothed SMR = 0.55. Why does smoothing *increase* spatial autocorrelation?
5. Implement a 2D kernel density smoothed risk surface from the point-level CVD rate data.


In [ ]:
# Exercise 5 — KDE risk surface
from scipy.stats import gaussian_kde

# Create a 2D KDE of CVD rates weighted by population
weights_pop = df.population / df.population.sum()
lon_arr = df.lon.values; lat_arr = df.lat.values

# Weighted KDE
stacked = np.vstack([lon_arr, lat_arr])
kde = gaussian_kde(stacked, weights=weights_pop * df.smr, bw_method=0.3)

# Evaluate on grid
lon_grid = np.linspace(lon_arr.min(), lon_arr.max(), 60)
lat_grid = np.linspace(lat_arr.min(), lat_arr.max(), 30)
LON, LAT = np.meshgrid(lon_grid, lat_grid)
grid_pts = np.vstack([LON.ravel(), LAT.ravel()])
kde_surface = kde(grid_pts).reshape(LON.shape)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.contourf(LON, LAT, kde_surface, levels=20, cmap="YlOrRd")
ax.contour(LON, LAT, kde_surface, levels=10, colors="gray", alpha=0.4, linewidths=0.8)
plt.colorbar(im, ax=ax, label="Population-weighted CVD risk density")
ax.scatter(lon_arr, lat_arr, c="white", s=10, alpha=0.4, zorder=3)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("KDE Risk Surface: Population-Weighted CVD Mortality Density")
plt.tight_layout(); plt.savefig("/tmp/mod07/kde_risk_surface.png", bbox_inches="tight"); plt.show()


---
**Next → NB-05: Spatial Cluster Detection (SaTScan-style)**